# Example beginner cheminformatics exercise (Python): interactive scatterplot with molecule-image tooltips

Below is a self-contained exercise you can give to beginners. It uses RDKit to compute simple molecular properties and Plotly to build an interactive scatterplot where hovering a point shows a 2D image of the molecule.

Learning goals
You will learn how to:
-	Load molecules from SMILES
-	Compute basic descriptors (MolWt, logP, TPSA)
-	Render 2D structures to PNG
-	Embed those images into interactive hover tooltips on a scatterplot


--------------


1) Setup
Install dependencies (in many environments, RDKit is easiest via conda; if you use pip, use whatever is standard in your org’s environment).

Suggested search queries (no links): “install rdkit conda”, “plotly python install”.

You’ll need an environment with:
-	rdkit
-	pandas
-	plotly
-	Pillow (often pulled in already, but safe to have)

-------------------


2) The exercise code

The code includes includes a small toy dataset of drug-like molecules.

When you hover over any point, the tooltip shows:

-	The compound name
-	Key properties (MolWt, logP, TPSA)

Can you add A 2D depiction of that molecule to the hover?

This is a practical pattern used a lot in exploratory chemistry dashboards.


In [4]:
# dashboard code to play with

import base64
from io import BytesIO

import pandas as pd
import plotly.express as px

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, Draw


# ----------------------------
# A) Toy dataset (SMILES)
# ----------------------------
data = [
    {"name": "Aspirin", "smiles": "CC(=O)OC1=CC=CC=C1C(=O)O"},
    {"name": "Caffeine", "smiles": "Cn1cnc2n(C)c(=O)n(C)c(=O)c12"},
    {"name": "Paracetamol", "smiles": "CC(=O)NC1=CC=C(O)C=C1O"},
    {"name": "Ibuprofen", "smiles": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"},
    {"name": "Naproxen", "smiles": "COC1=CC=CC2=C1C(C)C(C(=O)O)=C2"},
    {"name": "Lidocaine", "smiles": "CCN(CC)C(=O)C1=NC(C)C(C)=CC1N(CC)CC"},
    {"name": "Nicotine", "smiles": "CN1CCC[C@H]1c2cccnc2"},
    {"name": "Benzocaine", "smiles": "CCOC(=O)C1=CC=CC=C1N"},
    {"name": "Morphine", "smiles": "CN1CCC23C4C1CC(O)C2(C=C(C3O)O)C5=CC=CC=C45"},
    {"name": "Warfarin", "smiles": "CC(=O)CC(C1=CC=CC=C1)C(O)C2=CC=CC=C2C(=O)O"},
]

df = pd.DataFrame(data)


# ----------------------------
# B) RDKit: parse + descriptors
# ----------------------------
def smiles_to_mol(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    Chem.rdDepictor.Compute2DCoords(mol)
    return mol

df["mol"] = df["smiles"].apply(smiles_to_mol)
df = df[df["mol"].notna()].copy()

df["MolWt"] = df["mol"].apply(Descriptors.MolWt)
df["logP"] = df["mol"].apply(Crippen.MolLogP)
df["TPSA"] = df["mol"].apply(rdMolDescriptors.CalcTPSA)


# ----------------------------
# C) Make molecule images and embed in hover tooltips
#    Approach: render RDKit image -> base64 PNG -> <img src="data:...">
# ----------------------------
def mol_to_base64_png(mol, size=(220, 180)) -> str:
    img = Draw.MolToImage(mol, size=size)  # PIL image
    bio = BytesIO()
    img.save(bio, format="PNG")
    b64 = base64.b64encode(bio.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{b64}"

df["img_src"] = df["mol"].apply(mol_to_base64_png)

# Build an HTML hover snippet. Plotly will render this on hover.
df["hover_html"] = df.apply(
    lambda r: (
        f"<b>{r['name']}</b><br>"
        f"SMILES: {r['smiles']}<br>"
        f"MolWt: {r['MolWt']:.1f}<br>"
        f"logP: {r['logP']:.2f}<br>"
        f"TPSA: {r['TPSA']:.1f}<br>"
        f"<img src='{r['img_src']}' style='width:220px;border:1px solid #ccc;'/>"
    ),
    axis=1
)


# ----------------------------
# D) Plotly scatter with interactive hover
# ----------------------------
fig = px.scatter(
    df,
    x="MolWt",
    y="logP",
    color="TPSA",
    color_continuous_scale="Viridis",
    title="Cheminformatics: MolWt vs logP (hover for structure)",
    labels={"MolWt": "Molecular Weight", "logP": "cLogP", "TPSA": "Topological Polar Surface Area"},
)



fig.update_layout(
    hoverlabel=dict(bgcolor="white", font_size=12),
)

fig.show()


4) Beginner-friendly extension ideas (optional)

Try any one of these:

-	Change axes to TPSA vs logP and color by HBD/HBA.
-	Add a filter step (e.g., “keep MolWt < 500 and logP < 5”) and replot.
-	Compute Morgan fingerprints and add a second plot such as 2D PCA (still hover-to-see-structure).
